In [1]:
from facenet_pytorch import InceptionResnetV1
import torch
from torch import nn
import numpy as np

/home/parashift/FaceRecogition/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class FaceNetEmbedding(nn.Module):
    def __init__(self, original_model):
        super(FaceNetEmbedding, self).__init__()
        # Copy all layers except the final logits layer
        self.conv2d_1a = original_model.conv2d_1a
        self.conv2d_2a = original_model.conv2d_2a
        self.conv2d_2b = original_model.conv2d_2b
        self.maxpool_3a = original_model.maxpool_3a
        self.conv2d_3b = original_model.conv2d_3b
        self.conv2d_4a = original_model.conv2d_4a
        self.conv2d_4b = original_model.conv2d_4b
        self.repeat_1 = original_model.repeat_1
        self.mixed_6a = original_model.mixed_6a
        self.repeat_2 = original_model.repeat_2
        self.mixed_7a = original_model.mixed_7a
        self.repeat_3 = original_model.repeat_3
        self.block8 = original_model.block8
        self.avgpool_1a = original_model.avgpool_1a
        self.dropout = original_model.dropout
        self.last_linear = original_model.last_linear
        self.last_bn = original_model.last_bn
        
    def forward(self, x):
        x = self.conv2d_1a(x)
        x = self.conv2d_2a(x)
        x = self.conv2d_2b(x)
        x = self.maxpool_3a(x)
        x = self.conv2d_3b(x)
        x = self.conv2d_4a(x)
        x = self.conv2d_4b(x)
        x = self.repeat_1(x)
        x = self.mixed_6a(x)
        x = self.repeat_2(x)
        x = self.mixed_7a(x)
        x = self.repeat_3(x)
        x = self.block8(x)
        x = self.avgpool_1a(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.last_linear(x)
        x = self.last_bn(x)
        return x

def convert_facenet_to_onnx(model, onnx_output_path):
    """
    Convert InceptionResNetV1 model to ONNX format, keeping only the embedding output
    
    Args:
        model_path: Path to saved PyTorch FaceNet model
        onnx_output_path: Path where ONNX model will be saved
    """
    # Load the model
    model = model.to('cpu')
    model.eval()
    
    # Create embedding-only model
    # embedding_model = FaceNetEmbedding(model)
    # embedding_model.eval()
    
    # Create dummy input (160x160 is the standard input size for FaceNet)
    dummy_input = torch.randn(1, 3, 160, 160)
    
    # Test forward pass before export
    with torch.no_grad():
        test_output = model(dummy_input)
        print(f"Test output shape before export: {test_output.shape}")
        # Should print torch.Size([1, 512])
    
    # Export to ONNX
    torch.onnx.export(model,
                      dummy_input,
                      onnx_output_path,
                      export_params=True,
                      opset_version=11,
                      do_constant_folding=True,
                      input_names=['input'],
                      output_names=['embedding'],
                      dynamic_axes={'input': {0: 'batch_size'},
                                  'embedding': {0: 'batch_size'}})

In [8]:
checkpoint_path = "outputs/facenet_arcface-unpg_output/best_model.pt"
onnx_output_path = "facenet_casia_finetune_160x160.onnx"

In [9]:
print(f"Resuming from checkpoint: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path)


model = InceptionResnetV1(classify=False)

Resuming from checkpoint: outputs/facenet_arcface-unpg_output/best_model.pt


In [5]:
embedding_model = FaceNetEmbedding(model)

In [21]:
class ArcFaceWrapper(nn.Module):
    """Wrapper for FaceNet model to use with ArcFace loss"""
    def __init__(self, base_model, num_classes):
        super(ArcFaceWrapper, self).__init__()
        self.base_model = base_model
        self.num_classes = num_classes
    
    def forward(self, x):
        """Extract feature embeddings"""
        embeddings = self.base_model(x)
        return embeddings

model = ArcFaceWrapper(embedding_model, num_classes=1878)

In [24]:
checkpoint.keys()

odict_keys(['conv2d_1a.conv.weight', 'conv2d_1a.bn.weight', 'conv2d_1a.bn.bias', 'conv2d_1a.bn.running_mean', 'conv2d_1a.bn.running_var', 'conv2d_1a.bn.num_batches_tracked', 'conv2d_2a.conv.weight', 'conv2d_2a.bn.weight', 'conv2d_2a.bn.bias', 'conv2d_2a.bn.running_mean', 'conv2d_2a.bn.running_var', 'conv2d_2a.bn.num_batches_tracked', 'conv2d_2b.conv.weight', 'conv2d_2b.bn.weight', 'conv2d_2b.bn.bias', 'conv2d_2b.bn.running_mean', 'conv2d_2b.bn.running_var', 'conv2d_2b.bn.num_batches_tracked', 'conv2d_3b.conv.weight', 'conv2d_3b.bn.weight', 'conv2d_3b.bn.bias', 'conv2d_3b.bn.running_mean', 'conv2d_3b.bn.running_var', 'conv2d_3b.bn.num_batches_tracked', 'conv2d_4a.conv.weight', 'conv2d_4a.bn.weight', 'conv2d_4a.bn.bias', 'conv2d_4a.bn.running_mean', 'conv2d_4a.bn.running_var', 'conv2d_4a.bn.num_batches_tracked', 'conv2d_4b.conv.weight', 'conv2d_4b.bn.weight', 'conv2d_4b.bn.bias', 'conv2d_4b.bn.running_mean', 'conv2d_4b.bn.running_var', 'conv2d_4b.bn.num_batches_tracked', 'repeat_1.0.bran

In [10]:
# Load model state
model.load_state_dict(checkpoint)

<All keys matched successfully>

In [9]:
# Convert model
embedding_model = convert_facenet_to_onnx(embedding_model, onnx_output_path)

Test output shape before export: torch.Size([1, 512])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime

In [49]:
class FaceNetEmbeddingExport(nn.Module):
    """Wrapper class for exporting FaceNet embeddings to ONNX"""
    def __init__(self, model):
        super(FaceNetEmbeddingExport, self).__init__()
        self.model = model
        
    def forward(self, x):
        # If using ArcFace wrapper
        if hasattr(self.model, 'base_model'):
            embeddings = self.model.base_model(x)
        else:
            embeddings = self.model(x)
            
        # Normalize embeddings (L2 norm)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings
    

def export_model_to_onnx(model, save_path, input_shape=(1, 3, 112, 112)):
    """
    Export FaceNet model to ONNX format
    
    Args:
        model: Trained FaceNet model
        save_path: Path to save ONNX model
        input_shape: Input tensor shape (batch_size, channels, height, width)
    """
    # Set model to evaluation mode
    model.eval()
    
    # Create wrapper for export
    export_model = FaceNetEmbeddingExport(model)
    
    # Create dummy input tensor
    dummy_input = torch.randn(input_shape, requires_grad=True)
    
    # Export the model
    torch.onnx.export(
        export_model,               # model being run
        dummy_input,                # model input (or a tuple for multiple inputs)
        save_path,                  # where to save the model
        export_params=True,         # store the trained parameter weights inside the model file
        opset_version=12,           # the ONNX version to export the model to
        do_constant_folding=True,   # whether to execute constant folding for optimization
        input_names=['input'],      # the model's input names
        output_names=['embedding'], # the model's output names
        dynamic_axes={
            'input': {0: 'batch_size'},     # variable length axes
            'embedding': {0: 'batch_size'}
        }
    )
    
    print(f"Model exported to {save_path}")
    
    # Verify the model
    onnx_model = onnx.load(save_path)
    onnx.checker.check_model(onnx_model)
    print("ONNX model verified successfully")
    
    return save_path

In [50]:
export_model_to_onnx(model, "facenet_arcface-unpg_112x112.onnx", input_shape=(1, 3, 112, 112))

Model exported to facenet_arcface-unpg_112x112.onnx
ONNX model verified successfully


'facenet_arcface-unpg_112x112.onnx'

In [42]:
def test_onnx_model(onnx_path, test_image_path=None, input_shape=(1, 3, 112, 112)):
    """
    Test ONNX model with a sample image or random input
    
    Args:
        onnx_path: Path to the exported ONNX model
        test_image_path: Path to test image (optional)
        input_shape: Input tensor shape if not using test image
    """
    # Create ONNX Runtime session
    session = onnxruntime.InferenceSession(onnx_path)
    
    # Get input name
    input_name = session.get_inputs()[0].name
    
    # Prepare input
    if test_image_path:
        from PIL import Image
        from torchvision import transforms
        
        # Define preprocessing
        preprocess = transforms.Compose([
            transforms.Resize((112, 112)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
        
        # Load and preprocess image
        img = Image.open(test_image_path).convert('RGB')
        img_tensor = preprocess(img).unsqueeze(0).numpy()
    else:
        # Use random input
        img_tensor = np.random.randn(*input_shape).astype(np.float32)
    
    # Run inference
    outputs = session.run(None, {input_name: img_tensor})
    embedding = outputs[0]
    
    print(f"Embedding shape: {embedding.shape}")
    print(f"Embedding L2 norm: {np.linalg.norm(embedding)}")  # Should be close to 1.0
    
    return embedding

In [ ]:
test_onnx_model("facenet_casia_finetune_112x112.onnx")

In [5]:
model = InceptionResnetV1(classify=False)

In [15]:
print(model)

InceptionResnetV1(
  (conv2d_1a): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_2a): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_2b): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (maxpool_3a): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2d_3b): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_4a): 

In [3]:
print(next(model.parameters()).dtype)

torch.float32


In [ ]:
def load_model(model_path, device=None):
    """
    Load a trained FaceNet model from a .pt file
    
    Args:
        model_path: Path to the .pt model file
        device: Device to load the model on ('cpu' or 'cuda')
        
    Returns:
        model: Loaded FaceNet model
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Initialize a new FaceNet model
    model = InceptionResnetV1(pretrained=None, classify=False).to(device)
    
    # Load the state dict
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    
    # Set model to evaluation mode
    model.eval()
    
    print(f"Loaded model from {model_path} to {device}")
    return model

In [13]:
checkpoint_path = "outputs/facenet-512_arcface-unpg_500_epochs/last_model.pt"
onnx_output_path = "facenet_casia_arcface-unpg_160x160(test).onnx"

In [14]:
embedding_model = load_model(checkpoint_path)

Loaded model from outputs/facenet-512_arcface-unpg_500_epochs/last_model.pt to cuda


In [15]:
embedding_model = convert_facenet_to_onnx(embedding_model, onnx_output_path)

Test output shape before export: torch.Size([1, 512])


In [1]:
import torch
print(torch.version.cuda)  # Should match your installed CUDA version
print(torch.__version__)

12.4
2.6.0+cu124
